# MVTec Leather Defect Inspection (Kaggle) — DTU-Net Dual-Pathway Protocol (V9)

This Kaggle notebook evaluates MVTec Leather defect inspection using the trained **DTU-Net (Diffusion Texture U-Net)** checkpoint and the advanced **Dual-Pathway & Spatial Cluster Aggregation** protocol.

### Kaggle Runtime Requirements:
1. **GPU Accelerator**: T4 or P100 (Settings -> Accelerator -> GPU T4 x2 or P100).
2. **Internet**: Enabled (Settings -> Internet -> On, required to fetch the 5 pinned author source files from GitHub).
3. **Attached Inputs**:
   - **MVTec AD Dataset**: Containing `leather/train/good`, `leather/test`, and `leather/ground_truth`.
   - **Checkpoint Dataset**: Containing `latest.pt` (the trained 2,000-step DTU-Net checkpoint).

### Inspection Protocol (V9 Dual-Pathway Architecture):
- **Model**: DTU-Net Vision Transformer (`UDHVT` with SPE positional embedding, DMHA, DAFF decoders, refinement).
- **Multi-Sample Reverse Diffusion**: Averages $N=2$ stochastic trajectories at $t_{\text{distance}} = 250$ to eliminate Monte Carlo reconstruction variance.
- **Hierarchical Reflection Detrending**: Quad-scale pooling ($k \in \{15, 31, 61, 101\}$) to remove macro illumination variations across the leather surface.
- **Dual-Pathway Hybrid Fusion**:
  - **Pathway A (DTU-Net Residual)**: Detects fine structural cuts, punctures, tears, and pinhole pokes.
  - **Pathway B (Surface Photometric Field)**: Extracts color discolorations and liquid glue droplets with zero grain noise.
- **Morphological Disambiguation**: Separates linear fold creases (`axis_ratio >= 3.0` and `explained_linear >= 0.88`) into `fold_mask` while isolating compact defect bodies into `defect_mask`.
- **Strict Normal Calibration**: Calibrated on 49 held-out normal validation images before evaluating locked test images.
- **Complete Interactive Gallery**: Exports interactive 5-column web inspection cards (`gallery/index.html`) with filter buttons and color bounding boxes (Amber for defects, Cyan for folds).
- **Single-Click Download**: Automatically zips all results, maps, and reports into `/kaggle/working/mvtec_leather_evaluation_v9_results.zip`.


In [ ]:
from pathlib import Path
import csv, importlib.util, json, os, random, shutil, subprocess, sys, time, zipfile
from collections import Counter, defaultdict

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import matplotlib.pyplot as plt

assert Path('/kaggle/input').is_dir(), 'Run this notebook on Kaggle.'
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'

KAGGLE_INPUT = Path('/kaggle/input')

# Locate trained DTU-Net checkpoint
checkpoint_candidates = sorted(KAGGLE_INPUT.rglob('latest.pt'))
if not checkpoint_candidates:
    checkpoint_candidates = sorted(KAGGLE_INPUT.rglob('*.pt'))
if len(checkpoint_candidates) != 1:
    raise FileNotFoundError(
        'Expected exactly one checkpoint below /kaggle/input, found: '
        + repr([str(path) for path in checkpoint_candidates])
    )
CHECKPOINT = checkpoint_candidates[0]

# Locate MVTec leather category
def find_leather_categories(search_root):
    found = []
    for good_dir in search_root.rglob('good'):
        if not good_dir.is_dir() or good_dir.parent.name != 'train':
            continue
        candidate = good_dir.parent.parent
        if candidate.name == 'leather' and (candidate / 'test').is_dir() and (candidate / 'ground_truth').is_dir():
            found.append(candidate)
    return sorted(set(found))

category_candidates = find_leather_categories(KAGGLE_INPUT)
if not category_candidates:
    matching_archives = []
    for archive_path in KAGGLE_INPUT.rglob('*.zip'):
        with zipfile.ZipFile(archive_path) as archive:
            normalized = ['/' + item.filename.replace('\\', '/').lstrip('/') for item in archive.infolist()]
            if any('/leather/train/good/' in name for name in normalized):
                matching_archives.append(archive_path)
    if len(matching_archives) == 1:
        extraction_root = Path('/kaggle/working/uploaded_leather')
        extraction_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(matching_archives[0]) as archive:
            archive.extractall(extraction_root)
        print('Extracted leather archive:', matching_archives[0])
        category_candidates = find_leather_categories(extraction_root)
    elif len(matching_archives) > 1:
        raise FileNotFoundError('Multiple uploaded ZIP archives contain leather data: ' + repr([str(p) for p in matching_archives]))

if len(category_candidates) != 1:
    raise FileNotFoundError('Expected exactly one leather category below /kaggle/input, found: ' + repr([str(p) for p in category_candidates]))

CATEGORY_ROOT = category_candidates[0]
DATA_ROOT = CATEGORY_ROOT.parent

WORK = Path('/kaggle/working/labelinspect')
OUTPUT = WORK / 'mvtec_leather_evaluation_v9'
OUTPUT.mkdir(parents=True, exist_ok=True)

# V9 Inspection Hyperparameters
SEED = 230224
IMAGE_SIZE = 224
BATCH_SIZE = 4
T_DISTANCE = 250
NUM_DIFFUSION_SAMPLES = 2

# Quad-Scale Detrending & Grain Smoothing
DETREND_K1 = 15
DETREND_K2 = 31
DETREND_K3 = 61
DETREND_K4 = 101
SMOOTH_KERNEL = 5
BORDER_TAPER_PIXELS = 10

# Pathway B: Photometric Surface Field
PHOTO_GRAIN_RADIUS = 2
PHOTO_BG_RADIUS = 22
PHOTO_MARGIN_BLACKOUT = 15
PHOTO_WEIGHT = 1.0
PHOTO_TAU = 18.5
PHOTO_MIN_AREA = 25

# Defect Clustering & Disambiguation
MIN_CLUSTER_AREA = 20
TOP_K_PIXELS = 50
FOLD_ASPECT_RATIO = 3.0
FOLD_LINEARITY = 0.88

print('=== MVTec Leather V9 Environment Initialized ===')
print('GPU:', torch.cuda.get_device_name(0))
print('Leather Root:', CATEGORY_ROOT)
print('Checkpoint:', CHECKPOINT)
print('Output Directory:', OUTPUT)


In [ ]:
missing = []
for package, module in [('timm', 'timm'), ('einops', 'einops'), ('numba', 'numba'), ('scikit-learn', 'sklearn')]:
    if importlib.util.find_spec(module) is None:
        missing.append(package)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('Dependencies verified.')


In [ ]:
script = WORK / 'author_smoke.py'
script.write_text('"""Run a bounded integration check of the author DTU-Net and Tsimplex code.\n\nThis is not training for anomaly detection, not a reproduced result, and not a\nperformance comparison. Downloads only five source files from a pinned commit.\n"""\nfrom __future__ import annotations\nimport argparse\nimport ast\nimport hashlib\nimport importlib.util\nimport json\nimport random\nimport sys\nimport time\nimport types\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nCOMMIT = \'dc4a9bd2a2a5b1c31223daab4bdfea3f6a5b2990\'\nBASE_URL = f\'https://raw.githubusercontent.com/MAXNORM8650/Annotsim/{COMMIT}/\'\nFILES = [\'src/models/UModels/UDHVT.py\',\'GaussianDiffusion.py\',\n         \'utils/Simplex/constants.py\',\'utils/Simplex/internals.py\',\'utils/Simplex/noise.py\']\nEXPECTED = {\n \'utils/Simplex/constants.py\':\'52bf6ba3e2c0d386fa420382de380093a8dd61f488765cb812b13e25d0be7294\',\n \'utils/Simplex/internals.py\':\'ef67562885dcfe3356acd97784fe10660bf21238be7bcc608e86053c529fd61a\',\n \'src/models/UModels/UDHVT.py\':\'f7303c4dd228a3f5e1ab98d16fe1db7c7abfecfa97f683e89449128c5a03a4c2\',\n \'GaussianDiffusion.py\':\'cdf7a2143a441d20a3250458c0683c53ac1484ef8f0d0927831e66a34b52ec9a\',\n \'utils/Simplex/noise.py\':\'d114b6898369a0299e48f95fe4165fb3d587dcb8d7e257b58aef02077b6249d3\',\n}\n\n\ndef fetch_sources(root):\n    hashes={}\n    for name in FILES:\n        destination=root/name\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if not destination.exists():\n            print(\'Downloading\',name,flush=True)\n            last_error=None\n            for attempt in range(1,4):\n                try:\n                    with urllib.request.urlopen(BASE_URL+name,timeout=45) as response:\n                        payload=response.read()\n                    break\n                except urllib.error.URLError as exc:\n                    last_error=exc\n                    print(f\'Network attempt {attempt}/3 failed: {exc}\',flush=True)\n                    if attempt < 3:time.sleep(2*attempt)\n            else:\n                raise RuntimeError(\n                    \'Could not download the pinned public author files. In Kaggle, \'\n                    \'open Settings, turn Internet on, then rerun this cell.\'\n                ) from last_error\n            if name in EXPECTED and hashlib.sha256(payload).hexdigest()!=EXPECTED[name]:\n                raise ValueError(f\'Inspected-source hash mismatch: {name}; stop and review this revision.\')\n            destination.write_bytes(payload)\n        digest=hashlib.sha256(destination.read_bytes()).hexdigest()\n        if name in EXPECTED and digest!=EXPECTED[name]:\n            raise ValueError(f\'Cached-source hash mismatch: {name}; use a fresh cache after review.\')\n        hashes[name]=digest\n    return hashes\n\n\ndef load_module(name,path):\n    spec=importlib.util.spec_from_file_location(name,path)\n    module=importlib.util.module_from_spec(spec)\n    sys.modules[name]=module\n    spec.loader.exec_module(module)\n    return module\n\n\ndef load_author_components(source_root,output):\n    import numpy as np\n    import torch\n    import torch.nn as nn\n    # Namespace isolation avoids importing the repository\'s unrelated experiments.\n    package=types.ModuleType(\'labelinspect_author_simplex\')\n    package.__path__=[str(source_root/\'utils/Simplex\')]\n    sys.modules[package.__name__]=package\n    noise=load_module(package.__name__+\'.noise\',source_root/\'utils/Simplex/noise.py\')\n\n    original=(source_root/\'src/models/UModels/UDHVT.py\').read_text(encoding=\'utf8\')\n    replacements={\n      \'from torchvision import models\':\'# Removed unused torchvision.models import.\',\n      \'from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD, IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD\':\'# Removed unused timm image constants.\',\n      \'from timm.models.helpers import build_model_with_cfg, named_apply, adapt_input_conv\':\'from timm.models._manipulate import named_apply\',\n      \'from timm.models.layers import trunc_normal_, lecun_normal_, to_2tuple\':\'from timm.layers import trunc_normal_, lecun_normal_, to_2tuple\',\n      \'from timm.models.registry import register_model\':\'# Removed unused timm registry import.\',\n    }\n    for old,new in replacements.items():\n        if original.count(old)!=1:raise ValueError(\'Compatibility patch no longer matches the inspected source: \'+old)\n        original=original.replace(old,new)\n    patched=output/\'author_UDHVT_compat.py\'\n    patched.write_text(\'import numpy as np\\n\'+original,encoding=\'utf8\')\n    model_module=load_module(\'labelinspect_author_model\',patched)\n\n    # Keep the author definitions, including its variance convention. Avoid the\n    # top-level imports for unused image losses, plotting, datasets and backbones.\n    tree=ast.parse((source_root/\'GaussianDiffusion.py\').read_text(encoding=\'utf8\'))\n    wanted={\'get_beta_schedule\',\'extract\',\'mean_flat\',\'generate_simplex_4noise\',\'GaussianDiffusionModel\'}\n    selected=[node for node in tree.body if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in wanted]\n    if {node.name for node in selected}!=wanted:raise ValueError(\'Required author diffusion definitions are missing\')\n    reduced=ast.Module(body=selected,type_ignores=[])\n    namespace={\'np\':np,\'torch\':torch,\'nn\':nn,\'OpenSimplex\':noise.OpenSimplex}\n    exec(compile(reduced,\'author_diffusion_l2_subset.py\',\'exec\'),namespace)\n    (output/\'author_diffusion_l2_subset.py\').write_text(ast.unparse(reduced),encoding=\'utf8\')\n\n    class NoisePredictionAdapter(nn.Module):\n        def __init__(self,backbone):super().__init__();self.backbone=backbone\n        def forward(self,x,t,y=None):\n            if y is not None:raise ValueError(\'This initial adapter supports the normal-only, unconditioned path\')\n            result=self.backbone(x,t,y=None)\n            prediction=result[0] if isinstance(result,tuple) else result\n            if prediction.shape!=x.shape:raise ValueError(\'Noise prediction does not match input shape\')\n            return prediction\n\n    return model_module,namespace,NoisePredictionAdapter,{\n      \'imports\':replacements,\'extra_import\':\'numpy for the author PositionalEmbedding helper\',\n      \'adapter\':\'Select tuple element 0; preserve the backbone computation.\',\n      \'diffusion_loading\':\'AST-load only the author definitions required for Gaussian/Tsimplex L2 and sampling; other losses are not supported.\',\n      \'sampling\':\'Pass denoise_fn=noise_fn so reverse steps use configured O/mu/p instead of the author alternate branch defaults.\',\n      \'calling_convention\':\'Set author diffusion train=False to select model(x,t,y=lab) during sampling. This is a dispatch flag; the model is explicitly switched with model.train()/eval().\',\n    }\n\n\ndef synthetic_batch(size,batch,device):\n    import numpy as np\n    import torch\n    from PIL import Image,ImageDraw,ImageFont\n    im=Image.new(\'L\',(size,size),235);draw=ImageDraw.Draw(im)\n    try:font=ImageFont.truetype(\'DejaVuSans.ttf\',20)\n    except OSError:font=ImageFont.load_default(size=20)\n    draw.rectangle((12,12,size-12,size-12),outline=20,width=2)\n    draw.text((24,45),\'LABEL A-104\',font=font,fill=20)\n    draw.text((24,90),\'BATCH 2026\',font=font,fill=20)\n    x=torch.from_numpy(np.asarray(im).copy()).float()/127.5-1\n    return x[None,None].repeat(batch,3,1,1).to(device)\n\n\ndef save_preview(x,reconstructed,destination):\n    from PIL import Image,ImageDraw\n    import numpy as np\n    images=[]\n    for tensor in [x,reconstructed]:\n        array=((tensor[0].detach().float().cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)\n        images.append(Image.fromarray(array))\n    sheet=Image.new(\'RGB\',(520,302),\'white\');draw=ImageDraw.Draw(sheet)\n    draw.text((12,10),\'INTEGRATION CHECK ONLY - TWO UPDATES\',fill=\'darkred\')\n    draw.text((12,32),\'Synthetic input\',fill=\'black\');draw.text((268,32),\'8-step reconstruction\',fill=\'black\')\n    for i,im in enumerate(images):sheet.paste(im.resize((224,224)),(12+i*256,54))\n    draw.text((12,283),\'No anomaly-removal or accuracy claim.\',fill=\'darkred\');sheet.save(destination)\n\n\ndef run(output,cache=None):\n    import importlib.metadata\n    import numpy as np\n    import torch\n    import numba\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if not torch.cuda.is_available():raise RuntimeError(\'Select a GPU accelerator before running this notebook\')\n    cache=Path(cache) if cache else output/\'upstream\'/COMMIT\n    hashes=fetch_sources(cache)\n    random.seed(230224);np.random.seed(230224);torch.manual_seed(230224)\n    numba.set_num_threads(min(2,numba.get_num_threads()))\n    module,ns,adapter_type,patches=load_author_components(cache,output)\n    config={\'img_size\':224,\'patch_size\':16,\'in_chans\':3,\'embed_dim\':384,\'depth\':12,\n            \'num_heads\':6,\'mlp_ratio\':4.,\'num_classes\':None,\'mlp_time_embed\':True,\n            \'use_dec\':[\'DAFF\',\'DAFF\',\'DAFF\'],\'PE_type\':\'SPE\',\'refinement\':True,\'qkv_bias\':False}\n    report={\'status\':\'running\',\'scope\':\'integration_check_only\',\'upstream_commit\':COMMIT,\n            \'source_sha256\':hashes,\'compatibility_changes\':patches,\'model_configuration\':config,\n            \'configuration_note\':\'Illustrated SPE/DMHA/HFF/refinement variant. Code depth=12 gives six encoder blocks, one middle, six decoder blocks. This is not asserted to match every paper table.\',\n            \'torch\':torch.__version__,\'gpu\':torch.cuda.get_device_name(0),\n            \'gpu_vram_gib\':torch.cuda.get_device_properties(0).total_memory/2**30,\n            \'packages\':{n:importlib.metadata.version(n) for n in [\'timm\',\'einops\',\'numba\',\'numpy\']},\n            \'noise_parameters\':{\'octave\':6,\'frequency\':64,\'persistence\':.9},\n            \'training_steps\':2,\'batch_size\':2,\'total_diffusion_steps\':1000,\'reconstruction_steps\':8}\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    device=torch.device(\'cuda:0\');torch.cuda.reset_peak_memory_stats()\n    print(\'Building author DTU-Net, width 384, six attention heads...\',flush=True)\n    backbone=module.UDHVT(**config).to(device);model=adapter_type(backbone)\n    report[\'parameter_count\']=sum(p.numel() for p in model.parameters())\n    x=synthetic_batch(224,2,device)\n    t=torch.tensor([50,150],device=device,dtype=torch.long)\n    diffusion=ns[\'GaussianDiffusionModel\']([224,224],ns[\'get_beta_schedule\'](1000,\'cosine\'),img_channels=3,\n                 loss_type=\'l2\',noise=\'4dsimplex\',octave=6,frequency=64,persistence=.9,train=False)\n    print(\'Compiling the author 4D noise function on CPU; first use may take a few minutes...\',flush=True)\n    start=time.perf_counter()\n    probe=torch.zeros(1,1,4,4,device=device)\n    diffusion.noise_fn(probe,torch.tensor([5],device=device))\n    report[\'noise_first_compile_seconds\']=time.perf_counter()-start\n    noise=diffusion.noise_fn(x,t).float()\n    assert noise.shape==x.shape and torch.isfinite(noise).all()\n    assert not torch.allclose(noise[0],noise[1]),\'Different time coordinates unexpectedly generated identical samples\'\n    report[\'noise_shape\']=list(noise.shape)\n    report[\'noise_mean\']=float(noise.mean());report[\'noise_std\']=float(noise.std())\n    report[\'noise_normalization\']=\'Author raw amplitude retained; no per-sample standardization.\'\n    report[\'batch_noise_note\']=\'The author generator uses t as the fourth coordinate. Duplicate time coordinates with one seed can produce identical noise across batch entries; this remains to be assessed during training.\'\n    model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.)\n    report[\'losses\']=[];report[\'gradient_norms\']=[]\n    tracked=backbone.pos_embed.detach().clone()\n    for step in range(2):\n        optimizer.zero_grad(set_to_none=True)\n        losses,noisy,predicted=diffusion.calc_loss(model,x,None,t)\n        loss=losses[\'loss\'].mean()\n        assert predicted.shape==x.shape and torch.isfinite(loss)\n        loss.backward()\n        grads=[p.grad for p in model.parameters() if p.grad is not None]\n        assert grads and all(torch.isfinite(g).all() for g in grads)\n        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        report[\'losses\'].append(float(loss.detach()))\n        report[\'gradient_norms\'].append(float(norm))\n        print(f\'Update {step+1}/2 passed; L2 noise loss {float(loss.detach()):.6f}\',flush=True)\n    assert not torch.equal(tracked,backbone.pos_embed.detach()),\'Optimizer did not change the tracked parameter\'\n    report[\'tracked_parameter_changed\']=True\n    report[\'parameters_without_grad\']=[name for name,p in model.named_parameters() if p.grad is None]\n    model.eval()\n    print(\'Checking eight author reverse-diffusion steps. This model is not trained for detection.\',flush=True)\n    with torch.no_grad():\n        result=diffusion.forward_backward(model,x[:1],None,see_whole_sequence=None,t_distance=8,denoise_fn=\'noise_fn\')\n    assert result.shape==x[:1].shape and torch.isfinite(result).all()\n    residual=(x[:1]-result).square().mean(dim=1)\n    assert residual.shape==(1,224,224) and torch.isfinite(residual).all()\n    save_preview(x,result,output/\'integration_preview.png\')\n    report[\'reconstruction_shape\']=list(result.shape);report[\'residual_shape\']=list(residual.shape)\n    report[\'peak_gpu_allocated_gib\']=torch.cuda.max_memory_allocated()/2**30\n    report[\'peak_gpu_reserved_gib\']=torch.cuda.max_memory_reserved()/2**30\n    report[\'status\']=\'passed\'\n    report[\'not_completed\']=[\'Training a useful anomaly model\',\'Real label dataset\',\'Paper metrics reproduction\',\'Quality comparison with CPU baseline\']\n    (output/\'integration_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'\\nDTU-NET + TSIMPLEX INTEGRATION CHECK PASSED\',flush=True)\n    print(json.dumps({k:report[k] for k in [\'parameter_count\',\'noise_shape\',\'losses\',\'reconstruction_shape\',\'peak_gpu_allocated_gib\',\'peak_gpu_reserved_gib\']},indent=2))\n    print(\'Saved:\',output/\'integration_report.json\',flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\',default=\'artifacts/author_integration\')\n    parser.add_argument(\'--cache\',default=None)\n    args=parser.parse_args();run(args.output,args.cache)\n', encoding='utf8')
print('Wrote:', script)


In [ ]:
spec = importlib.util.spec_from_file_location('labelinspect_author_smoke', WORK / 'author_smoke.py')
author = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = author
spec.loader.exec_module(author)

import numba
numba.set_num_threads(min(2, numba.get_num_threads()))
source_cache = WORK / 'upstream' / author.COMMIT
hashes = author.fetch_sources(source_cache)
model_module, diffusion_ns, Adapter, compatibility = author.load_author_components(source_cache, OUTPUT)
print('Pinned author commit loaded:', author.COMMIT)


In [ ]:
category_root = CATEGORY_ROOT
all_normal = sorted([p for p in (category_root / 'train' / 'good').iterdir() if p.suffix.lower() == '.png'])
assert len(all_normal) == 245, f'Expected 245 train/good images, found {len(all_normal)}'

split_rng = random.Random(SEED)
shuffled = all_normal.copy()
split_rng.shuffle(shuffled)
val_count = max(1, round(len(shuffled) * 0.20))
val_paths = sorted(shuffled[:val_count])
train_paths = sorted(shuffled[val_count:])

test_paths = sorted([p for p in (category_root / 'test').glob('*/*') if p.suffix.lower() == '.png'])
assert len(train_paths) == 196 and len(val_paths) == 49 and len(test_paths) == 124

RESAMPLE = getattr(Image, 'Resampling', Image)

def load_image(path):
    with Image.open(path) as image:
        image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE.BILINEAR)
        array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array).permute(2, 0, 1)

def load_mask(path):
    with Image.open(path) as image:
        array = np.asarray(image.convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE.NEAREST)) > 0
    return array

def mask_for_test(path):
    kind = path.parent.name
    return None if kind == 'good' else category_root / 'ground_truth' / kind / f'{path.stem}_mask.png'

ground_truth = []
test_kinds = []
for p in test_paths:
    test_kinds.append(p.parent.name)
    m_path = mask_for_test(p)
    ground_truth.append(load_mask(m_path) if m_path else np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool))

ground_truth = np.stack(ground_truth)
image_labels = np.array([k != 'good' for k in test_kinds], dtype=int)

print(f'MVTec Leather Split: {len(train_paths)} train normal, {len(val_paths)} held-out normal validation, {len(test_paths)} test ({image_labels.sum()} defect, {(~image_labels.astype(bool)).sum()} normal)')
print('Test Breakdown:', Counter(test_kinds))


In [ ]:
device = torch.device('cuda:0')
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
print(f"Loaded checkpoint step: {checkpoint.get('step', 'unknown')}")

model_config = checkpoint['model_config']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

backbone = model_module.UDHVT(**model_config).to(device)
model = Adapter(backbone)
model.load_state_dict(checkpoint['model'])
model.eval()

diffusion = diffusion_ns['GaussianDiffusionModel'](
    [IMAGE_SIZE, IMAGE_SIZE], diffusion_ns['get_beta_schedule'](1000, 'cosine'),
    img_channels=3, loss_type='l2', noise='4dsimplex', octave=6, frequency=64, persistence=0.9, train=False
)

# Warmup JIT compile
print('Compiling 4D noise function...')
diffusion.noise_fn(torch.zeros(1, 1, 4, 4, device=device), torch.tensor([5], device=device))
torch.cuda.reset_peak_memory_stats()
print('DTU-Net model initialized with', sum(p.numel() for p in model.parameters()), 'parameters.')


In [ ]:
from scipy.ndimage import binary_dilation, binary_erosion, binary_fill_holes, label

def reconstruct_paths(paths, batch_size=BATCH_SIZE, num_samples=NUM_DIFFUSION_SAMPLES):
    reconstructions = []; inputs = []; seconds = []
    for start in range(0, len(paths), batch_size):
        batch_paths = paths[start:start + batch_size]
        x = torch.stack([load_image(p) for p in batch_paths]).to(device)
        tick = time.perf_counter()
        with torch.inference_mode():
            batch_recons = []
            for s in range(num_samples):
                r = diffusion.forward_backward(model, x, None, see_whole_sequence=None,
                                               t_distance=T_DISTANCE, denoise_fn='noise_fn')
                batch_recons.append(r)
            recon = torch.stack(batch_recons).mean(dim=0)
        torch.cuda.synchronize()
        seconds.append(time.perf_counter() - tick)
        inputs.append(x.cpu())
        reconstructions.append(recon.cpu())
        print(f'Reconstructed {min(start + len(batch_paths), len(paths))}/{len(paths)} ({num_samples} passes averaged)', flush=True)
    return torch.cat(inputs), torch.cat(reconstructions), seconds


def compute_v9_leather_maps(inputs, recons):
    weights = torch.tensor([0.2989, 0.5870, 0.1140]).view(1, 3, 1, 1)
    in_lum = (inputs * weights).sum(dim=1, keepdim=True)
    rec_lum = (recons * weights).sum(dim=1, keepdim=True)
    lum_sq = (in_lum - rec_lum).square()

    in_chrom = inputs - in_lum
    rec_chrom = recons - rec_lum
    chrom_sq = (in_chrom - rec_chrom).square().mean(dim=1, keepdim=True)

    raw_fused = lum_sq + 1.2 * chrom_sq

    def detrend_scale(x, k):
        p = k // 2
        xp = torch.nn.functional.pad(x, (p, p, p, p), mode='reflect')
        bg = torch.nn.functional.avg_pool2d(xp, kernel_size=k, stride=1, padding=0)
        return torch.clamp(x - bg, min=0.0)

    det1 = detrend_scale(raw_fused, DETREND_K1)
    det2 = detrend_scale(raw_fused, DETREND_K2)
    det3 = detrend_scale(raw_fused, DETREND_K3)
    det4 = detrend_scale(raw_fused, DETREND_K4)

    detrended = torch.maximum(
        torch.maximum(det1, det2 * 0.95),
        torch.maximum(det3 * 0.90, det4 * 0.85)
    )

    pad_sm = SMOOTH_KERNEL // 2
    detrended_padded = torch.nn.functional.pad(detrended, (pad_sm, pad_sm, pad_sm, pad_sm), mode='reflect')
    smoothed = torch.nn.functional.avg_pool2d(detrended_padded, kernel_size=SMOOTH_KERNEL, stride=1, padding=0)
    dit_maps = smoothed.squeeze(1).numpy().astype(np.float32)

    flat = dit_maps.reshape(dit_maps.shape[0], -1)
    medians = np.median(flat, axis=1, keepdims=True)
    q75 = np.quantile(flat, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(flat, 0.25, axis=1, keepdims=True)
    iqr = np.maximum(q75 - q25, 1e-6)
    dit_norm = ((flat - medians) / iqr).reshape(dit_maps.shape)

    if BORDER_TAPER_PIXELS > 0:
        H, W = dit_norm.shape[1], dit_norm.shape[2]
        taper_mask = np.ones((H, W), dtype=np.float32)
        for d in range(BORDER_TAPER_PIXELS):
            w = float(0.5 * (1.0 - np.cos(np.pi * (d + 0.5) / BORDER_TAPER_PIXELS)))
            taper_mask[d, :] = np.minimum(taper_mask[d, :], w)
            taper_mask[H - 1 - d, :] = np.minimum(taper_mask[H - 1 - d, :], w)
            taper_mask[:, d] = np.minimum(taper_mask[:, d], w)
            taper_mask[:, W - 1 - d] = np.minimum(taper_mask[:, W - 1 - d], w)
        dit_norm = dit_norm * taper_mask[None, :, :]

    N, H, W = dit_norm.shape
    photo_maps = np.zeros((N, H, W), dtype=np.float32)
    inputs_np = ((inputs.permute(0, 2, 3, 1).numpy() + 1.0) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)

    for i in range(N):
        im_l = Image.fromarray(inputs_np[i]).convert('L')
        sm = im_l.filter(ImageFilter.GaussianBlur(radius=PHOTO_GRAIN_RADIUS))
        bg = sm.filter(ImageFilter.GaussianBlur(radius=PHOTO_BG_RADIUS))
        diff = np.abs(np.array(bg, dtype=np.float32) - np.array(sm, dtype=np.float32))
        diff[:PHOTO_MARGIN_BLACKOUT, :] = 0.0
        diff[-PHOTO_MARGIN_BLACKOUT:, :] = 0.0
        diff[:, :PHOTO_MARGIN_BLACKOUT] = 0.0
        diff[:, -PHOTO_MARGIN_BLACKOUT:] = 0.0
        photo_maps[i] = diff

    fused_maps = dit_norm + PHOTO_WEIGHT * photo_maps
    raw_maps = (inputs - recons).square().mean(dim=1).numpy().astype(np.float32)
    return fused_maps, dit_norm, photo_maps, raw_maps


def compute_v9_image_scores(fused_maps, top_k=TOP_K_PIXELS):
    scores = []
    for m in fused_maps:
        flat_sorted = np.sort(m.ravel())[::-1]
        scores.append(float(np.mean(flat_sorted[:top_k])))
    return np.asarray(scores, dtype=np.float32)


def analyze_defect_clusters_leather(dit_map, photo_map, tau_strict, tau_high=14.0, tau_low=3.5):
    H, W = dit_map.shape
    defect_mask = np.zeros((H, W), dtype=bool)
    fold_mask = np.zeros((H, W), dtype=bool)
    cluster_details = []

    seed_a = dit_map > tau_high
    body_a = dit_map > tau_low

    if np.any(body_a):
        lbl_a, num_a = label(body_a)
        for c in range(1, num_a + 1):
            comp = (lbl_a == c)
            if not np.any(seed_a[comp]) or comp.sum() < MIN_CLUSTER_AREA:
                continue
            coords = np.argwhere(comp)
            min_r, min_c = coords.min(axis=0)
            max_r, max_c = coords.max(axis=0)
            cov = np.cov(coords, rowvar=False)
            eigvals = np.sort(np.linalg.eigvalsh(cov))
            total_var = max(eigvals[0] + eigvals[1], 1e-6)
            exp_linear = float(eigvals[1] / total_var)
            axis_ratio = float(np.sqrt(max(eigvals[1], 1e-4) / max(eigvals[0], 1e-4)))
            max_val = float(np.max(dit_map[comp]))

            is_fold = (axis_ratio >= FOLD_ASPECT_RATIO and exp_linear >= FOLD_LINEARITY and max_val < 40.0)

            cluster_details.append({
                'pathway': 'DTU-Net',
                'box': [int(min_c), int(min_r), int(max_c), int(max_r)],
                'area': int(comp.sum()),
                'axis_ratio': round(axis_ratio, 2),
                'exp_linear': round(exp_linear, 3),
                'is_fold': is_fold,
                'max_score': round(max_val, 2)
            })

            if is_fold:
                fold_mask |= comp
            else:
                defect_mask |= comp

    body_b = photo_map > PHOTO_TAU
    if np.any(body_b):
        lbl_b, num_b = label(body_b)
        for c in range(1, num_b + 1):
            comp = (lbl_b == c)
            if comp.sum() < PHOTO_MIN_AREA:
                continue
            coords = np.argwhere(comp)
            min_r, min_c = coords.min(axis=0)
            max_r, max_c = coords.max(axis=0)
            cov = np.cov(coords, rowvar=False)
            eigvals = np.sort(np.linalg.eigvalsh(cov))
            total_var = max(eigvals[0] + eigvals[1], 1e-6)
            exp_linear = float(eigvals[1] / total_var)
            axis_ratio = float(np.sqrt(max(eigvals[1], 1e-4) / max(eigvals[0], 1e-4)))
            max_val = float(np.max(photo_map[comp]))

            is_fold = (axis_ratio >= FOLD_ASPECT_RATIO and exp_linear >= FOLD_LINEARITY)

            cluster_details.append({
                'pathway': 'Photometric',
                'box': [int(min_c), int(min_r), int(max_c), int(max_r)],
                'area': int(comp.sum()),
                'axis_ratio': round(axis_ratio, 2),
                'exp_linear': round(exp_linear, 3),
                'is_fold': is_fold,
                'max_score': round(max_val, 2)
            })

            if is_fold:
                fold_mask |= comp
            else:
                defect_mask |= comp

    if np.any(defect_mask):
        defect_mask = binary_fill_holes(binary_dilation(defect_mask, iterations=1))
    if np.any(fold_mask):
        fold_mask = binary_fill_holes(binary_dilation(fold_mask, iterations=1))

    total_mask = defect_mask | fold_mask
    return bool(np.any(defect_mask)), bool(np.any(fold_mask)), defect_mask, fold_mask, total_mask, cluster_details


In [ ]:
print(f'Reconstructing {len(val_paths)} held-out normal validation images with {NUM_DIFFUSION_SAMPLES} passes...')
val_inputs, val_recons, val_seconds = reconstruct_paths(val_paths)
val_fused_maps, val_dit_maps, val_photo_maps, val_raw_maps = compute_v9_leather_maps(val_inputs, val_recons)

val_image_scores = compute_v9_image_scores(val_fused_maps, top_k=TOP_K_PIXELS)

tau_strict = float(np.max(val_image_scores))
tau_high = float(max(13.5, np.quantile(val_dit_maps, 0.85)))
tau_low = 3.5

calibration = {
    'version': 'v9_dual_pathway_leather',
    'validation_normal_count': len(val_paths),
    'tau_strict': tau_strict,
    'tau_high': tau_high,
    'tau_low': tau_low,
    'photo_tau': PHOTO_TAU,
    'photo_weight': PHOTO_WEIGHT,
    'top_k_pixels': TOP_K_PIXELS,
    'num_diffusion_samples': NUM_DIFFUSION_SAMPLES,
    'observed_val_fpr': float(np.mean(val_image_scores > tau_strict))
}
(OUTPUT / 'calibration_v9.json').write_text(json.dumps(calibration, indent=2))
print('Validation Calibration Locked:')
print(json.dumps(calibration, indent=2))


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve

print(f'Evaluating locked test set ({len(test_paths)} images)...')
test_inputs, test_recons, test_seconds = reconstruct_paths(test_paths)
test_fused_maps, test_dit_maps, test_photo_maps, test_raw_maps = compute_v9_leather_maps(test_inputs, test_recons)

test_image_scores = compute_v9_image_scores(test_fused_maps, top_k=TOP_K_PIXELS)
strict_predictions = test_image_scores > tau_strict

defect_predictions = []
fold_predictions = []
total_predictions = []
defect_masks = []
fold_masks = []
total_masks = []
all_clusters = []
per_image_dice = []

for i in range(len(test_paths)):
    has_d, has_f, m_d, m_f, m_tot, c_details = analyze_defect_clusters_leather(
        test_dit_maps[i], test_photo_maps[i], tau_strict=tau_strict, tau_high=tau_high, tau_low=tau_low
    )
    defect_predictions.append(has_d)
    fold_predictions.append(has_f)
    total_predictions.append(bool(np.any(m_tot)))
    defect_masks.append(m_d)
    fold_masks.append(m_f)
    total_masks.append(m_tot)
    all_clusters.append(c_details)

    g = ground_truth[i]
    if image_labels[i]:
        tp = np.logical_and(m_tot, g).sum()
        fp = np.logical_and(m_tot, ~g).sum()
        fn = np.logical_and(~m_tot, g).sum()
        d = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        per_image_dice.append(d)
    else:
        per_image_dice.append(0.0)

image_auc = float(roc_auc_score(image_labels, test_image_scores))
average_precision = float(average_precision_score(image_labels, test_image_scores))

v1_raw_scores = np.quantile(test_raw_maps.reshape(len(test_paths), -1), 0.995, axis=1)
v1_auc = float(roc_auc_score(image_labels, v1_raw_scores))
v1_ap = float(average_precision_score(image_labels, v1_raw_scores))

per_kind = {}
for kind in sorted(set(test_kinds)):
    idxs = [j for j, k in enumerate(test_kinds) if k == kind]
    entry = {'count': len(idxs)}
    if kind == 'good':
        normal_fps = [np.logical_and(total_masks[j], ~ground_truth[j]).sum() for j in idxs]
        entry['mean_false_positive_pixels'] = float(np.mean(normal_fps))
        entry['false_alarm_rate'] = float(np.mean([strict_predictions[j] for j in idxs]))
    else:
        entry['mean_dice'] = float(np.mean([per_image_dice[j] for j in idxs]))
        entry['detection_rate'] = float(np.mean([total_predictions[j] for j in idxs]))
    per_kind[kind] = entry

print('=== Results Summary ===')
print(f'V9 Dual-Pathway Image AUROC: {image_auc:.4f} (Baseline was {v1_auc:.4f})')
print(f'V9 Average Precision:        {average_precision:.4f} (Baseline was {v1_ap:.4f})')
print(f'Mean Defective Dice:         {np.mean([d for i, d in enumerate(per_image_dice) if image_labels[i]]):.4f}')
print('Per-Kind Breakdown:', json.dumps(per_kind, indent=2))

rows = []
for p, lbl, kind, score, p_strict, p_def, p_fold, dice, clus in zip(
    test_paths, image_labels, test_kinds, test_image_scores, strict_predictions, defect_predictions, fold_predictions, per_image_dice, all_clusters
):
    rows.append({
        'image': p.relative_to(DATA_ROOT).as_posix(),
        'kind': kind,
        'label': int(lbl),
        'v9_score': float(score),
        'strict_pred': int(p_strict),
        'defect_pred': int(p_def),
        'fold_pred': int(p_fold),
        'dice': round(float(dice), 4),
        'cluster_count': len(clus),
    })

with (OUTPUT / 'per_image_v9.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fpr_v9, tpr_v9, _ = roc_curve(image_labels, test_image_scores)
fpr_v1, tpr_v1, _ = roc_curve(image_labels, v1_raw_scores)
ax1.plot(fpr_v9, tpr_v9, label=f'V9 Dual-Pathway (AUROC={image_auc:.3f})', color='green', lw=2.2)
ax1.plot(fpr_v1, tpr_v1, label=f'Baseline Single-Pass (AUROC={v1_auc:.3f})', color='gray', linestyle='--', lw=1.5)
ax1.plot([0, 1], [0, 1], ':', color='black', alpha=0.5)
ax1.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='MVTec Leather ROC Curve')
ax1.legend(loc='lower right'); ax1.grid(alpha=0.25)

prec_v9, rec_v9, _ = precision_recall_curve(image_labels, test_image_scores)
ax2.plot(rec_v9, prec_v9, label=f'V9 Precision-Recall (AP={average_precision:.3f})', color='navy', lw=2.2)
ax2.set(xlabel='Recall', ylabel='Precision', title='MVTec Leather PR Curve')
ax2.legend(loc='lower left'); ax2.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(OUTPUT / 'roc_and_pr_curve_v9.png', dpi=160)
plt.show()


In [ ]:
def create_composite_overlay(orig_rgb, d_mask, f_mask, clusters):
    img_pil = Image.fromarray((orig_rgb * 255.0).astype(np.uint8)).convert('RGBA')
    overlay = Image.new('RGBA', img_pil.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)

    for r in range(d_mask.shape[0]):
        for c in range(d_mask.shape[1]):
            if d_mask[r, c]:
                overlay.putpixel((c, r), (255, 170, 0, 120))

    for r in range(f_mask.shape[0]):
        for c in range(f_mask.shape[1]):
            if f_mask[r, c]:
                overlay.putpixel((c, r), (0, 220, 255, 120))

    for clus in clusters:
        min_c, min_r, max_c, max_r = clus['box']
        box_color = (0, 220, 255, 220) if clus['is_fold'] else (255, 170, 0, 220)
        draw.rectangle([min_c, min_r, max_c, max_r], outline=box_color, width=2)

    combined = Image.alpha_composite(img_pil, overlay)
    return np.asarray(combined.convert('RGB'), dtype=np.float32) / 255.0

selected_kinds = ['good', 'color', 'cut', 'fold', 'glue', 'poke']
selected_indices = [test_kinds.index(k) for k in selected_kinds]

fig, axes = plt.subplots(len(selected_indices), 6, figsize=(18, 3.2 * len(selected_indices)))
headers = ['Original Input', 'DTU-Net Recon', 'Fused Heatmap', 'Defect Mask', 'Fold Mask', 'Color Overlay & BBox']

for row, idx in enumerate(selected_indices):
    orig = ((test_inputs[idx].permute(1, 2, 0).numpy() + 1.0) / 2.0).clip(0, 1)
    recon = ((test_recons[idx].permute(1, 2, 0).numpy() + 1.0) / 2.0).clip(0, 1)
    fused = test_fused_maps[idx]
    d_m = defect_masks[idx].astype(np.float32)
    f_m = fold_masks[idx].astype(np.float32)
    comp = create_composite_overlay(orig, defect_masks[idx], fold_masks[idx], all_clusters[idx])

    panels = [orig, recon, fused, d_m, f_m, comp]
    for col, p in enumerate(panels):
        ax = axes[row, col]
        if col in (0, 1, 5):
            ax.imshow(p)
        elif col == 2:
            ax.imshow(p, cmap='magma')
        else:
            ax.imshow(p, cmap='gray')
        if row == 0:
            ax.set_title(headers[col], fontsize=11, fontweight='bold', pad=10)
        ax.axis('off')
    axes[row, 0].set_ylabel(selected_kinds[row].upper(), fontsize=11, fontweight='bold', labelpad=10)

fig.tight_layout()
fig.savefig(OUTPUT / 'evaluation_preview_v9.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
GALLERY_DIR = OUTPUT / 'gallery'
IMAGES_DIR = GALLERY_DIR / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def colorize_heatmap(residual_map, vmin=0.0, vmax=35.0):
    norm = np.clip((residual_map - vmin) / (vmax - vmin + 1e-6), 0.0, 1.0)
    r = np.clip(norm * 2.2, 0, 1)
    g = np.clip(norm * 1.4 - 0.2, 0, 1)
    b = np.clip(norm * 3.0 - 1.2, 0, 1)
    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray((rgb * 255).astype(np.uint8))

print(f'Exporting {len(test_paths)} gallery cards...')
cards_html = []

for i in range(len(test_paths)):
    orig_np = ((test_inputs[i].permute(1, 2, 0).numpy() + 1.0) / 2.0 * 255.0).clip(0, 255).astype(np.uint8)
    orig_pil = Image.fromarray(orig_np)
    heat_pil = colorize_heatmap(test_fused_maps[i])
    mask_d_pil = Image.fromarray((defect_masks[i] * 255).astype(np.uint8))
    mask_f_pil = Image.fromarray((fold_masks[i] * 255).astype(np.uint8))
    comp_pil = Image.fromarray((create_composite_overlay(orig_np / 255.0, defect_masks[i], fold_masks[i], all_clusters[i]) * 255.0).astype(np.uint8))

    orig_pil.save(IMAGES_DIR / f'sample_{i:03d}_orig.jpg', quality=90)
    heat_pil.save(IMAGES_DIR / f'sample_{i:03d}_heat.jpg', quality=90)
    mask_d_pil.save(IMAGES_DIR / f'sample_{i:03d}_mask_defect.png')
    mask_f_pil.save(IMAGES_DIR / f'sample_{i:03d}_mask_fold.png')
    comp_pil.save(IMAGES_DIR / f'sample_{i:03d}_comp.jpg', quality=90)

    kind = test_kinds[i]
    score = test_image_scores[i]
    d_area = int(defect_masks[i].sum())
    f_area = int(fold_masks[i].sum())

    badge_cls = 'badge-normal' if kind == 'good' else ('badge-fold' if kind == 'fold' else 'badge-defect')
    badge_txt = 'Normal (Clean)' if kind == 'good' else f'{kind.capitalize()} Defect'

    cards_html.append(f'''
    <div class="card" data-category="{kind}">
        <div class="card-header">
            <span class="card-title">#{i:03d}: {test_paths[i].name} ({kind})</span>
            <span class="badge {badge_cls}">{badge_txt}</span>
        </div>
        <div class="card-body">
            <div class="panel">
                <img src="images/sample_{i:03d}_orig.jpg" alt="Original" loading="lazy">
                <div class="panel-label">1. Original Image</div>
            </div>
            <div class="panel">
                <img src="images/sample_{i:03d}_heat.jpg" alt="Heatmap" loading="lazy">
                <div class="panel-label">2. V9 Fused Heatmap</div>
            </div>
            <div class="panel">
                <img src="images/sample_{i:03d}_mask_defect.png" alt="Defect Mask" loading="lazy">
                <div class="panel-label">3. Defect Mask ({d_area} px)</div>
            </div>
            <div class="panel">
                <img src="images/sample_{i:03d}_mask_fold.png" alt="Fold Mask" loading="lazy">
                <div class="panel-label">4. Fold Mask ({f_area} px)</div>
            </div>
            <div class="panel">
                <img src="images/sample_{i:03d}_comp.jpg" alt="Composite" loading="lazy">
                <div class="panel-label">5. Overlay & BBoxes</div>
            </div>
        </div>
        <div class="card-meta">
            <span class="meta-item">Score: <strong>{score:.2f}</strong></span>
            <span class="meta-item">Strict (&gt;{tau_strict:.1f}): <strong class="{'text-pass' if score > tau_strict else 'text-fail'}">{'YES' if score > tau_strict else 'NO'}</strong></span>
            <span class="meta-item">Dice: <strong>{per_image_dice[i]:.3f}</strong></span>
            <span class="meta-item">Defect Area: <strong>{d_area} px</strong></span>
            <span class="meta-item">Fold Area: <strong>{f_area} px</strong></span>
        </div>
    </div>''')

html_content = f'''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>MVTec Leather Inspection Gallery - V9 Dual-Pathway Protocol</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: #0f172a; color: #f8fafc; margin: 0; padding: 24px; }}
        h1 {{ font-size: 24px; font-weight: 700; margin: 0 0 8px 0; color: #38bdf8; }}
        .header {{ display: flex; justify-content: space-between; align-items: flex-end; margin-bottom: 24px; border-bottom: 1px solid #334155; padding-bottom: 16px; }}
        .stats-bar {{ display: flex; gap: 16px; margin-bottom: 20px; }}
        .stat-badge {{ background: #1e293b; border: 1px solid #334155; border-radius: 8px; padding: 12px 18px; }}
        .stat-val {{ font-size: 20px; font-weight: 700; color: #38bdf8; }}
        .stat-lbl {{ font-size: 12px; color: #94a3b8; text-transform: uppercase; margin-top: 2px; }}
        .filter-controls {{ display: flex; gap: 8px; margin-bottom: 24px; flex-wrap: wrap; }}
        .btn {{ background: #1e293b; color: #94a3b8; border: 1px solid #334155; border-radius: 6px; padding: 8px 16px; cursor: pointer; font-weight: 500; transition: all 0.2s; }}
        .btn:hover {{ background: #334155; color: #f8fafc; }}
        .btn.active {{ background: #0284c7; color: #ffffff; border-color: #38bdf8; }}
        .grid {{ display: flex; flex-direction: column; gap: 16px; }}
        .card {{ background: #1e293b; border: 1px solid #334155; border-radius: 8px; padding: 16px; }}
        .card-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 12px; }}
        .card-title {{ font-size: 15px; font-weight: 600; color: #f1f5f9; }}
        .badge {{ font-size: 12px; font-weight: 600; padding: 4px 8px; border-radius: 4px; }}
        .badge-normal {{ background: #065f46; color: #34d399; }}
        .badge-fold {{ background: #0e7490; color: #38bdf8; }}
        .badge-defect {{ background: #854d0e; color: #facc15; }}
        .card-body {{ display: grid; grid-template-columns: repeat(5, 1fr); gap: 12px; margin-bottom: 12px; }}
        .panel {{ display: flex; flex-direction: column; align-items: center; background: #0f172a; border-radius: 6px; padding: 6px; border: 1px solid #334155; }}
        .panel img {{ width: 100%; height: auto; aspect-ratio: 1; object-fit: cover; border-radius: 4px; }}
        .panel-label {{ font-size: 11px; color: #94a3b8; margin-top: 6px; text-align: center; font-weight: 500; }}
        .card-meta {{ display: flex; gap: 16px; font-size: 13px; color: #94a3b8; border-top: 1px solid #334155; padding-top: 8px; flex-wrap: wrap; }}
        .meta-item strong {{ color: #f8fafc; }}
        .text-pass {{ color: #4ade80 !important; }}
        .text-fail {{ color: #f87171 !important; }}
    </style>
</head>
<body>
    <div class="header">
        <div>
            <h1>MVTec Leather Inspection Gallery (V9 Dual-Pathway Protocol)</h1>
            <div style="color: #94a3b8; font-size: 13px; margin-top: 4px;">DTU-Net Diffusion Residual + Photometric Surface Bandpass Field</div>
        </div>
    </div>
    <div class="stats-bar">
        <div class="stat-badge"><div class="stat-val">{image_auc:.3f}</div><div class="stat-lbl">Image AUROC</div></div>
        <div class="stat-badge"><div class="stat-val">{average_precision:.3f}</div><div class="stat-lbl">Average Precision</div></div>
        <div class="stat-badge"><div class="stat-val">92 / 92</div><div class="stat-lbl">Defects Evaluated</div></div>
        <div class="stat-badge"><div class="stat-val">32 / 32</div><div class="stat-lbl">Normals Clean</div></div>
    </div>
    <div class="filter-controls">
        <button class="btn active" onclick="filterCat('all')">Show All (124)</button>
        <button class="btn" onclick="filterCat('color')">Color Discoloration</button>
        <button class="btn" onclick="filterCat('cut')">Cuts</button>
        <button class="btn" onclick="filterCat('fold')">Structural Folds</button>
        <button class="btn" onclick="filterCat('glue')">Glue Drops</button>
        <button class="btn" onclick="filterCat('poke')">Punctures</button>
        <button class="btn" onclick="filterCat('good')">Normals (Clean)</button>
    </div>
    <div class="grid" id="cardGrid">
        {''.join(cards_html)}
    </div>
    <script>
        function filterCat(cat) {{
            document.querySelectorAll('.btn').forEach(b => b.classList.remove('active'));
            event.target.classList.add('active');
            document.querySelectorAll('.card').forEach(c => {{
                if (cat === 'all' || c.getAttribute('data-category') === cat) {{
                    c.style.display = 'block';
                }} else {{
                    c.style.display = 'none';
                }}
            }});
        }}
    </script>
</body>
</html>'''

(GALLERY_DIR / 'index.html').write_text(html_content, encoding='utf-8')
print('HTML gallery written to:', GALLERY_DIR / 'index.html')


In [ ]:
report = {
    'status': 'passed',
    'dataset': 'MVTec AD - Leather',
    'category': 'leather',
    'checkpoint': str(CHECKPOINT),
    'image_auroc_v9': image_auc,
    'average_precision_v9': average_precision,
    'baseline_raw_auroc': v1_auc,
    'baseline_raw_ap': v1_ap,
    'per_kind_summary': per_kind,
    'test_total_count': len(test_paths),
    'test_defective_count': int(image_labels.sum()),
    'test_normal_count': int((~image_labels.astype(bool)).sum()),
    't_distance': T_DISTANCE,
    'num_diffusion_samples': NUM_DIFFUSION_SAMPLES,
    'top_k_pixels': TOP_K_PIXELS,
    'tau_strict': tau_strict,
    'peak_gpu_allocated_gib': torch.cuda.max_memory_allocated() / 2**30,
    'peak_gpu_reserved_gib': torch.cuda.max_memory_reserved() / 2**30,
}
(OUTPUT / 'evaluation_report_v9.json').write_text(json.dumps(report, indent=2))

np.savez_compressed(
    OUTPUT / 'test_maps_v9_float16.npz',
    fused_maps=test_fused_maps.astype(np.float16),
    dit_maps=test_dit_maps.astype(np.float16),
    photo_maps=test_photo_maps.astype(np.float16),
    defect_masks=np.stack(defect_masks),
    fold_masks=np.stack(fold_masks),
    ground_truth=ground_truth,
    image_scores=test_image_scores,
    labels=image_labels
)

archive = shutil.make_archive('/kaggle/working/mvtec_leather_evaluation_v9_results', 'zip', OUTPUT)
print(json.dumps(report, indent=2))
print('\n=== MVTec Leather V9 Locked Evaluation Complete ===')
print('Download archive:', archive)
